# DTM API Python Package - Example Usage

This notebook demonstrates the usage of the `dtmapi` package for accessing Displacement Tracking Matrix (DTM) data.

**Package Version:** 0.1.7

**Key Features:**
- Support for both API v3 (current) and v2 (legacy)
- Access to IDP data at Admin 0, 1, and 2 levels
- Gender disaggregation, origin of displacement, and displacement reason (v3 only)
- Built-in parameter validation and error handling
- Automatic retry logic for failed requests

## 1. Installation & Setup

Install the package:
```bash
pip install dtmapi
```

In [ ]:
# Import the package
from dtmapi import DTMApi
import pandas as pd

# Set your API subscription key
# Get your key from: https://dtm-apim-portal.iom.int/
SUBSCRIPTION_KEY = "YOUR-API-KEY-HERE"

# Initialize the API client (defaults to v3)
api = DTMApi(subscription_key=SUBSCRIPTION_KEY)

## 2. Get Available Countries and Operations

In [21]:
# Get all countries with available DTM data
countries = api.get_all_countries()
print(f"Total countries: {len(countries)}")
countries.head(10)

Total countries: 53


,admin0Name,admin0Pcode
0,Afghanistan,AFG
1,Antigua and Barbuda,ATG
2,Bahamas (the),BHS
3,Benin,BEN
4,Bolivia (Plurinational State of),BOL
5,Burkina Faso,BFA
6,Burundi,BDI
7,Cameroon,CMR
8,Central African Republic,CAF
9,Chad,TCD


In [22]:
# Get all operations
operations = api.get_all_operations()
print(f"Total operations: {len(operations)}")
operations.head(10)

Total operations: 106


,operation,operationStatus,admin0Name,admin0Pcode
0,Aceh earthquake,Inactive,Indonesia,IDN
1,Armed Clashes in Sudan,Active,Sudan,SDN
2,Armed Clashes in Sudan (Monthly),Active,Sudan,SDN
3,Armed Clashes in Sudan (Overview),Active,Sudan,SDN
4,Arrivals in Armenia,Inactive,Republic of Armenia,ARM
5,As-Sweida Conflict,Active,Syrian Arab Republic,SYR
6,Bahamas (the) - Hurricane Dorian Response,Inactive,Bahamas (the),BHS
7,Biometric Registration,Active,South Sudan,SSD
8,Burkina Faso Crisis,Inactive,Burkina Faso,BFA
9,Burundi Complex Emergency,Active,Burundi,BDI


## 3. IDP Admin 0 Data (Country Level)

Get country-level IDP data with filtering by rounds.

In [23]:
# Get IDP data for Sudan, rounds 1-10
admin0_data = api.get_idp_admin0_data(
    CountryName='Sudan',
    FromRoundNumber=1,
    ToRoundNumber=10
)

print(f"Retrieved {len(admin0_data)} records")
print(f"Columns: {list(admin0_data.columns)}")
admin0_data.head()

Retrieved 382 records
Columns: ['operation', 'admin0Name', 'admin0Pcode', 'numPresentIdpInd', 'reportingDate', 'yearReportingDate', 'monthReportingDate', 'roundNumber', 'displacementReason', 'numberMales', 'numberFemales', 'idpOriginAdmin1Name', 'idpOriginAdmin1Pcode', 'assessmentType']


,operation,admin0Name,admin0Pcode,numPresentIdpInd,reportingDate,yearReportingDate,monthReportingDate,roundNumber,displacementReason,numberMales,numberFemales,idpOriginAdmin1Name,idpOriginAdmin1Pcode,assessmentType
0,Darfur conflict,Sudan,SDN,10311,2010-06-30T00:00:00,2010,6,1,Conflict,NaN,NaN,Not available,Not available,BA
1,Darfur conflict,Sudan,SDN,32766,2011-02-28T00:00:00,2011,2,2,Conflict,NaN,NaN,Not available,Not available,BA
2,Darfur conflict,Sudan,SDN,89616,2011-03-30T00:00:00,2011,3,3,Conflict,NaN,NaN,Not available,Not available,BA
3,Darfur conflict,Sudan,SDN,402407,2011-09-30T00:00:00,2011,9,7,Conflict,NaN,NaN,Not available,Not available,BA
4,Darfur conflict,Sudan,SDN,465280,2011-10-30T00:00:00,2011,10,8,Conflict,NaN,NaN,Not available,Not available,BA


### Explore Gender Disaggregation (v3 Feature)

In [24]:
# Check for gender-related columns (v3 only)
gender_cols = [col for col in admin0_data.columns if 'male' in col.lower() or 'female' in col.lower()]
print(f"Gender-related columns: {gender_cols}")

if gender_cols:
    # Display gender breakdown
    admin0_data[['operation', 'admin0Name', 'roundNumber'] + gender_cols].head()

Gender-related columns: ['numberMales', 'numberFemales']


## 4. IDP Admin 1 Data (State/Province Level)

Get state/province-level data with date filtering.

In [25]:
# Get IDP Admin 1 data for Sudan with date range
admin1_data = api.get_idp_admin1_data(
    CountryName='Sudan',
    FromReportingDate='2020-01-01',
    ToReportingDate='2024-08-15'
)

print(f"Retrieved {len(admin1_data)} records")
admin1_data.head()

Retrieved 4303 records


,id,operation,admin0Name,admin0Pcode,admin1Name,admin1Pcode,numPresentIdpInd,reportingDate,yearReportingDate,monthReportingDate,roundNumber,displacementReason,numberMales,numberFemales,idpOriginAdmin1Name,idpOriginAdmin1Pcode,assessmentType
0,2201,Darfur conflict,Sudan,SDN,West Kordofan,SD18,19,2020-01-30T00:00:00,2020,1,1,Conflict,NaN,NaN,Central Darfur,SD06,BA
1,2202,Darfur conflict,Sudan,SDN,West Kordofan,SD18,3287,2020-01-30T00:00:00,2020,1,1,Conflict,NaN,NaN,East Darfur,SD05,BA
2,2203,Darfur conflict,Sudan,SDN,West Kordofan,SD18,3795,2020-01-30T00:00:00,2020,1,1,Conflict,NaN,NaN,North Darfur,SD02,BA
3,2204,Darfur conflict,Sudan,SDN,West Kordofan,SD18,775,2020-01-30T00:00:00,2020,1,1,Conflict,NaN,NaN,South Darfur,SD03,BA
4,2205,Darfur conflict,Sudan,SDN,West Kordofan,SD18,2285,2020-01-30T00:00:00,2020,1,1,Conflict,NaN,NaN,South Kordofan,SD07,BA


### Explore Origin of Displacement (v3 Feature)

In [26]:
# Check for origin-related columns (v3 only)
origin_cols = [col for col in admin1_data.columns if 'origin' in col.lower()]
print(f"Origin-related columns: {origin_cols}")

if origin_cols:
    # Display origin information
    admin1_data[['admin0Name', 'admin1Name'] + origin_cols].drop_duplicates().head(10)

Origin-related columns: ['idpOriginAdmin1Name', 'idpOriginAdmin1Pcode']


## 5. IDP Admin 2 Data (District Level)

Get district-level data filtered by operation.

In [27]:
# Get IDP Admin 2 data for Sudan
admin2_data = api.get_idp_admin2_data(
    CountryName='Sudan',
    FromRoundNumber=1,
    ToRoundNumber=5
)

print(f"Retrieved {len(admin2_data)} records")
admin2_data.head()

Retrieved 5446 records


,id,operation,admin0Name,admin0Pcode,admin1Name,admin1Pcode,admin2Name,admin2Pcode,numPresentIdpInd,reportingDate,yearReportingDate,monthReportingDate,roundNumber,displacementReason,numberMales,numberFemales,idpOriginAdmin1Name,idpOriginAdmin1Pcode,assessmentType
0,89645,Darfur conflict,Sudan,SDN,West Darfur,SD04,Beida,SD04111,10311,2010-06-30T00:00:00,2010,6,1,Conflict,NaN,NaN,Not available,Not available,BA
1,43104,Darfur conflict,Sudan,SDN,West Darfur,SD04,Beida,SD04111,10311,2011-02-28T00:00:00,2011,2,2,Conflict,NaN,NaN,Not available,Not available,BA
2,48286,Darfur conflict,Sudan,SDN,North Darfur,SD02,Melit,SD02129,4818,2011-02-28T00:00:00,2011,2,2,Conflict,NaN,NaN,Not available,Not available,BA
3,67915,Darfur conflict,Sudan,SDN,Central Darfur,SD06,Mukjar,SD06130,17637,2011-02-28T00:00:00,2011,2,2,Conflict,NaN,NaN,Not available,Not available,BA
4,61285,Darfur conflict,Sudan,SDN,West Darfur,SD04,Beida,SD04111,10311,2011-03-30T00:00:00,2011,3,3,Conflict,NaN,NaN,Not available,Not available,BA


### Explore Displacement Reason (v3 Feature)

In [28]:
# Check for displacement reason column (v3 only)
if 'displacementReason' in admin2_data.columns:
    print("Displacement reasons:")
    print(admin2_data['displacementReason'].value_counts())
    
    # Display data by reason
    admin2_data.groupby('displacementReason')['numPresentIdpInd'].sum().sort_values(ascending=False)

Displacement reasons:
displacementReason
Conflict                               5307
Conflict; Natural disaster               75
No reason for displacement reported      28
Economic reasons                         21
Natural disaster                         14
Other reason                              1
Name: count, dtype: int64


## 6. API Version Comparison (v2 vs v3)

Compare the differences between API v2 (legacy) and v3 (current).

In [29]:
# Initialize both API versions
api_v3 = DTMApi(subscription_key=SUBSCRIPTION_KEY, api_version="v3")
api_v2 = DTMApi(subscription_key=SUBSCRIPTION_KEY, api_version="v2")

print("API clients initialized")
print(f"v3 client: {api_v3.api_version}")
print(f"v2 client: {api_v2.api_version}")

API clients initialized
v3 client: v3
v2 client: v2


In [30]:
# Fetch same data from both versions
data_v3 = api_v3.get_idp_admin0_data(CountryName='Sudan', FromRoundNumber=1, ToRoundNumber=5)
data_v2 = api_v2.get_idp_admin0_data(CountryName='Sudan', FromRoundNumber=1, ToRoundNumber=5)

print(f"v3 data: {len(data_v3)} records, {len(data_v3.columns)} columns")
print(f"v2 data: {len(data_v2)} records, {len(data_v2.columns)} columns")

v3 data: 222 records, 14 columns
v2 data: 23 records, 10 columns


In [31]:
# Compare columns
v3_cols = set(data_v3.columns)
v2_cols = set(data_v2.columns)

print("\nColumns ONLY in v3 (new features):")
v3_only = v3_cols - v2_cols
for col in sorted(v3_only):
    print(f"  - {col}")

print("\nCommon columns:")
common = v3_cols.intersection(v2_cols)
print(f"  Total: {len(common)} columns")


Columns ONLY in v3 (new features):
  - displacementReason
  - idpOriginAdmin1Name
  - idpOriginAdmin1Pcode
  - numberFemales
  - numberMales

Common columns:
  Total: 9 columns


### v3 Enhanced Features Summary

API v3 includes these additional fields:
- **Gender Disaggregation**: `numberMales`, `numberFemales`
- **Origin of Displacement**: `idpOriginAdmin1Name`, `idpOriginAdmin1Pcode`
- **Displacement Reason**: `displacementReason`

## 7. Data Analysis Examples

In [32]:
# Example: IDP data summary for Sudan
admin0_summary = api.get_idp_admin0_data(
    CountryName='Sudan',
    FromRoundNumber=1,
    ToRoundNumber=50
)

# Get latest data
latest_data = admin0_summary.sort_values('reportingDate').tail(10)

# Display latest records
print("Latest IDP Records for Sudan:")
latest_data[['operation', 'numPresentIdpInd', 'reportingDate', 'roundNumber']].head()

Latest IDP Records for Sudan:


,operation,numPresentIdpInd,reportingDate,roundNumber
1083,Armed Clashes in Sudan (Overview),28633,2025-09-30T00:00:00,28
1085,Armed Clashes in Sudan (Overview),7914,2025-09-30T00:00:00,28
1084,Armed Clashes in Sudan (Overview),15514,2025-09-30T00:00:00,28
1082,Armed Clashes in Sudan (Overview),230490,2025-09-30T00:00:00,28
1081,Armed Clashes in Sudan (Overview),334810,2025-09-30T00:00:00,28


In [33]:
# Example: Gender breakdown analysis (v3 only)
if 'numberMales' in admin0_summary.columns and 'numberFemales' in admin0_summary.columns:
    # Calculate total by gender
    gender_summary = admin0_summary.groupby('admin0Name')[['numberMales', 'numberFemales']].sum()
    gender_summary['Total'] = gender_summary['numberMales'] + gender_summary['numberFemales']
    gender_summary['% Female'] = (gender_summary['numberFemales'] / gender_summary['Total'] * 100).round(1)
    
    print("\nGender Breakdown by Country:")
    gender_summary.sort_values('Total', ascending=False).head(10)


Gender Breakdown by Country:


## 8. Error Handling

The package includes built-in error handling and validation.

In [34]:
# Example: Invalid date format (will raise ValidationError)
from dtmapi import ValidationError

try:
    data = api.get_idp_admin0_data(
        CountryName='Sudan',
        FromReportingDate='01-01-2024'  # Invalid format
    )
except ValidationError as e:
    print(f"Validation Error: {e}")

Validation Error: FromReportingDate must be in YYYY-MM-DD format, got: 01-01-2024


In [35]:
# Example: Invalid date range (will raise ValidationError)
try:
    data = api.get_idp_admin0_data(
        CountryName='Sudan',
        FromReportingDate='2024-12-31',
        ToReportingDate='2024-01-01'  # End before start
    )
except ValidationError as e:
    print(f"Validation Error: {e}")

Validation Error: FromReportingDate (2024-12-31) must be before or equal to ToReportingDate (2024-01-01)


In [36]:
# Example: Missing required parameters (will raise ValidationError)
try:
    data = api.get_idp_admin0_data()  # No filters provided
except ValidationError as e:
    print(f"Validation Error: {e}")

Validation Error: At least one of the following parameters is required: Operation, CountryName, Admin0Pcode


## 9. Advanced Configuration

Customize timeout and retry settings.

In [37]:
# Initialize with custom settings
api_custom = DTMApi(
    subscription_key=SUBSCRIPTION_KEY,
    api_version="v3",
    timeout=60,        # 60 second timeout
    max_retries=5,     # Retry up to 5 times
    retry_delay=2.0    # 2 second base delay between retries
)

print("Custom API client initialized")
print(f"Timeout: {api_custom.timeout}s")
print(f"Max retries: {api_custom.max_retries}")
print(f"Retry delay: {api_custom.retry_delay}s")

Custom API client initialized
Timeout: 60s
Max retries: 5
Retry delay: 2.0s


## 10. Export Data

Export the retrieved data to various formats.

In [38]:
# Get some data
export_data = api.get_idp_admin0_data(CountryName='Sudan', FromRoundNumber=1, ToRoundNumber=10)

# Export to CSV
export_data.to_csv('sudan_idp_data.csv', index=False)
print("Data exported to sudan_idp_data.csv")

# Export to Excel
export_data.to_excel('sudan_idp_data.xlsx', index=False)
print("Data exported to sudan_idp_data.xlsx")

# Export to JSON
export_data.to_json('sudan_idp_data.json', orient='records', indent=2)
print("Data exported to sudan_idp_data.json")

Data exported to sudan_idp_data.csv
Data exported to sudan_idp_data.xlsx
Data exported to sudan_idp_data.json


## Resources

- **Documentation**: https://dtmapi.readthedocs.io
- **GitHub**: https://github.com/Displacement-tracking-Matrix/dtmapi
- **DTM Website**: https://dtm.iom.int/
- **API Registration**: https://dtm-apim-portal.iom.int/

For questions or feedback: dtmdataconsolidation@iom.int